# Reproduce the <TOPIC> analysis — the "Verify" layer

This is the **reproducible notebook** that sits at the bottom of the Inspector /
coding-verifier layer. It re-runs, *from raw data*, the pipeline behind the
blog's headline numbers and **asserts** that the reproduced figures match the
published ones — so the notebook is a *proof*, not just a script.

**This notebook reproduces by DOWNLOAD-AND-RUN-LOCALLY.** Download it (the blog's
"Download notebook" link) together with the bundled `verify/data/` inputs and run
it locally — there is no Colab branch and no remote fetch.

## How to run

* Download this notebook **and** the `verify/data/` inputs shipped beside it.
* Run the cells top to bottom; `cell_setup` finds the dataset via the `DATA_DIR`
  resolver (it checks `<PREFIX>_DATA_DIR`, then repo-relative / absolute paths —
  the first that resolves wins). Set `<PREFIX>_DATA_DIR` if your data lives
  elsewhere.
* One knob (e.g. `N`): the published value reproduces numbers exactly; a smaller
  value runs fast and lands within noise.

> Read-only on the dataset and on the blog's `code/` scripts. Regenerated files
> go to `verify/_repro_out/`, never back into the dataset.

*(Generic placeholders throughout — replace `<TOPIC>` / `city_transit` /
`ridership.csv` with your dataset.)*


In [ ]:
# === Setup: imports, constants, DATA_DIR resolver, guarded reads ===
# Download-and-run-locally: the dataset inputs ship beside this notebook under
# verify/data/. The DATA_DIR resolver finds them on disk — there is NO Colab
# branch and no remote fetch.
import os, json, math
from pathlib import Path
import numpy as np
import pandas as pd

# --- ONE data-location constant; every read pulls from DATA_DIR. Local only. ---
_CANDIDATES = [
    os.environ.get("DEMO_DATA_DIR"),            # explicit override (env <PREFIX>_DATA_DIR)
    "verify/data",                               # inputs shipped beside this notebook
    "../../../../datasets/city_transit",         # repo-relative to this notebook
    "data/city_transit",                          # cwd = repo root
]
REQUIRED = ["ridership.csv", "outputs/published_summary.json"]   # sentinel = ridership.csv

def _resolve_data_dir(candidates):
    for cand in candidates:
        if not cand:
            continue
        p = Path(cand).expanduser().resolve()
        if (p / "ridership.csv").exists():       # sentinel input
            return p
    return None

# Resolve locally (first candidate that resolves wins). Fail LOUDLY with a hint
# instead of fetching — download the bundled verify/data/ inputs to reproduce.
DATA_DIR = _resolve_data_dir(_CANDIDATES)
assert DATA_DIR is not None, (
    "Could not locate the dataset. Download the bundled verify/data/ inputs next to "
    "this notebook, or set DEMO_DATA_DIR to the folder containing ridership.csv, "
    "then re-run."
)

# Guarded reads: assert every required input exists before using it.
for rel in REQUIRED:
    assert (DATA_DIR / rel).exists(), f"missing required input: {rel}"

# Model constants (mirror the source scripts exactly).
SEED = 12345
N = 100000          # set N smaller for a fast smoke run (within noise, not exact)

# Output dir for regenerated files (never write into the dataset).
REPRO_OUT = Path("_repro_out").resolve()
REPRO_OUT.mkdir(exist_ok=True)

print("DATA_DIR resolved to:", DATA_DIR)
print(f"knobs: N={N:,}  SEED={SEED}")
print("regenerated files (if any) ->", REPRO_OUT)


In [ ]:
# === Example compute cell (one per source script in code/) — re-express + self-assert ===
# Pattern: re-derive a PUBLISHED number from raw data, then assert it matches.
# This cell stands in for a real `cell_<script>` (e.g. cell_model / cell_findings).

published = json.loads((DATA_DIR / "outputs/published_summary.json").read_text(encoding="utf-8"))

# ridership.csv is the read-only input; recompute the published headline from it.
rides = pd.read_csv(DATA_DIR / "ridership.csv", encoding="utf-8")
repro_total = int(rides["riders"].sum())            # <-- the reproduced number

# Self-assert: the notebook is a PROOF — the reproduced number must match.
pub_total = int(published["total_riders"])
print(f"reproduced total riders = {repro_total:,}  (published {pub_total:,})")
assert repro_total == pub_total, "reproduced total does not match published outputs"
print("OK: matches the published number.")

# (Stochastic variant — grade a smaller sample within noise, never assert equal:)
# assert abs(repro_estimate - pub_value) < 0.01, "outside Monte-Carlo noise"


## Provenance summary & licenses

Every published number traces to a script + source data, all reproduced above
from `DATA_DIR`.

| Finding (blog) | Reproduced in | Source script | Source data |
|---|---|---|---|
| <headline number> | `cell_example` | `code/<script>.py` | `ridership.csv` |

### Licenses

* **<dataset>** — <license> — <what it provides>.

*Regenerated files from this notebook live in `verify/_repro_out/` and are never
written back into the read-only dataset.*
